# Improve RAG accuracy with fine-tuned embedding models

### https://aws.amazon.com/ko/blogs/machine-learning/improve-rag-accuracy-with-fine-tuned-embedding-models-on-amazon-sagemaker/

In [1]:
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the pre-trained SentenceTransformer model
# GPU가 가능하면 자동으로 GPU로 모델을 이동
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)


dataset=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv')
# Convert the dataset to the SentenceTransformer input format
def prepare_data(data, text_column1, text_column2):
    # 데이터에서 NaN 값을 빈 문자열로 대체하고, 비문자열을 제거
    data[text_column1] = data[text_column1].fillna('').astype(str)
    data[text_column2] = data[text_column2].fillna('').astype(str)

    input_examples = []
    for _, row in data.iterrows():
        text1 = row[text_column1]
        text2 = row[text_column2]
        input_examples.append(InputExample(texts=[text1, text2], label=1.0))

    return input_examples



/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
data=dataset[['question','text']]

In [3]:
data.head(320)

,question,text
0,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
3,When does the new bunk'd come out?,\n## Episodes\n\n### Season 1 (2015–16)\n
4,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
...,...,...
315,When does the new bunk'd come out?,07-12)715\nGuest stars: Jordan Clark as Victor...
316,When does the new bunk'd come out?,"719\nGuest stars: Thom Rivera as The Marshal, ..."
317,When does the new bunk'd come out?,"as Victoria, Brandilyn Cheah as Scout, Keanus..."
318,When does the new bunk'd come out?,hashi are in town to do a gymnast exposition a...


In [ ]:
# # SQuAD 같은 텍스트 데이터셋을 사용한 예시 (TriviaQA도 가능)
# squad_data = load_dataset("squad", split="train[:1000]")  # 예를 들어 1000개의 샘플만 사용
# train_examples = prepare_data(squad_data)

In [15]:
train_batch_size = 32
# 데이터 전처리
train_data = prepare_data(dataset, 'question', 'text')

# Train/Val split
train_df, val_df = train_test_split(dataset[['question', 'text']], test_size=0.3, random_state=42)

# 학습용 데이터셋 준비
train_examples = prepare_data(train_df, 'question', 'text')
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=train_batch_size)

# 검증용 데이터셋 준비
val_df["scores"] = 1.0
val_df.reset_index(inplace=True, drop= True)

In [14]:
val_df.head()

,question,text,scores
1634,When does doctor strange get the infinity stone?,overhear two sailors in a bar discussing a he...,1.0
5598,Where is the microtubules located in a cell?,vertical offset of 3 tubulin monomers due to ...,1.0
8496,Who owns the rights to masters of the universe?,"Heroes, which included a collection of action ...",1.0
12366,How long was the term for the Texas Governor b...,"–January 15, 1895(did not run);Democratic[18];...",1.0
795,Who won the war between ethiopia and italy?,\n#### Table: First Italo-Ethiopian War\nPart ...,1.0


In [16]:
# Loss 설정 (MultipleNegativesRankingLoss)
train_loss = losses.MultipleNegativesRankingLoss(model)

# Early Stopping을 위한 설정
patience = 3
best_val_loss = float('inf')
patience_counter = 0

# 검증을 위한 evaluator 설정 (EmbeddingSimilarityEvaluator 사용)
dev_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df["question"],
    sentences2=val_df["text"],
    scores=val_df["scores"],
    main_similarity=SimilarityFunction.COSINE,
    name="dev_eval",
)

# Train the model with GPU support
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,  # 조정 가능
    evaluator=dev_evaluator,
    show_progress_bar=True,
    use_amp=True  # 혼합 정밀도(FP16) 사용, 필요에 따라 생략 가능
)
# 학습된 모델 저장
model.save("fine_tuned_model")

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/torch/nn/parallel/data_parallel.py:34: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 4 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(imbalance_warn.format(device_ids[min_pos], device_ids[max_pos]))


Step,Training Loss,Validation Loss,Dev Eval Pearson Cosine,Dev Eval Spearman Cosine,Dev Eval Pearson Manhattan,Dev Eval Spearman Manhattan,Dev Eval Pearson Euclidean,Dev Eval Spearman Euclidean,Dev Eval Pearson Dot,Dev Eval Spearman Dot,Dev Eval Pearson Max,Dev Eval Spearman Max
62,No log,No log,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
124,No log,No log,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
186,No log,No log,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/evaluation/EmbeddingSimilarityEvaluator.py:192: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson_cosine, _ = pearsonr(labels, cosine_scores)
/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/evaluation/EmbeddingSimilarityEvaluator.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman_cosine, _ = spearmanr(labels, cosine_scores)
/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/evaluation/EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson_manhattan, _ = pearsonr(labels, manhattan_distances)
/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/evaluation/EmbeddingSimilarityEvaluator.py:196: Co

KeyboardInterrupt: 

In [31]:
data

,question,text
0,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
3,When does the new bunk'd come out?,\n## Episodes\n\n### Season 1 (2015–16)\n
4,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
...,...,...
14083,When did Star Wars the Last Jedi come out thro...,Williams;Nominated;[162]\nBest Original Score...
14084,When did Star Wars the Last Jedi come out thro...,2018;Favorite Movie;Star Wars: The Last Jedi;N...
14085,When did Star Wars the Last Jedi come out thro...,;[171]\nDaisy Ridley;Nominated\nChoice Fantasy...
14086,When did Star Wars the Last Jedi come out thro...,"Fujita, Jiyong Shin, and Dan Finnegan for ""Me..."


In [74]:
question=data['question'].drop_duplicates().tolist()
context=data['text'].tolist()

In [77]:
data

,question,text
0,When does the new bunk'd come out?,Bunk'd is an American comedy television series...
1,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
2,When does the new bunk'd come out?,\n#### Table: SeasonEpisodesOriginally aired\n...
3,When does the new bunk'd come out?,\n## Episodes\n\n### Season 1 (2015–16)\n
4,When does the new bunk'd come out?,\n#### Table: No.overallNo. inseasonTitle [1][...
...,...,...
14083,When did Star Wars the Last Jedi come out thro...,Williams;Nominated;[162]\nBest Original Score...
14084,When did Star Wars the Last Jedi come out thro...,2018;Favorite Movie;Star Wars: The Last Jedi;N...
14085,When did Star Wars the Last Jedi come out thro...,;[171]\nDaisy Ridley;Nominated\nChoice Fantasy...
14086,When did Star Wars the Last Jedi come out thro...,"Fujita, Jiyong Shin, and Dan Finnegan for ""Me..."


In [72]:
dataset_unique = data.drop_duplicates(subset='question', keep='first')
context2=dataset_unique['text'].tolist()
# 기존 리스트에서 빈 문자열('') 제거
context2 = [text for text in context2 if text.strip()]

In [73]:
context2

["Bunk'd is an American comedy television series created by Pamela Eells O'Connell that aired on Disney Channel from July 31, 2015 to August 2, 2024. The series is a spinoff of Jessie and includes returning stars Peyton List, Karan Brar, and Skai Jackson. Starring alongside them is Miranda May.\n\n## Series overview",
 "Capital punishment is a legal punishment in Pennsylvania. Despite remaining a legal penalty, there have been no executions in Pennsylvania since 1999, and only three since 1976 (all occurring in the 1990s, during the governorship of Tom Ridge). In February 2015, Governor Tom Wolf announced a formal moratorium on executions that is still in effect as of 2023[update], with incumbent Governor Josh Shapiro continuing Wolf's moratorium.[1] However, capital crimes are still prosecuted and death warrants are still issued.\n",
 "The Premier League is an English professional league for association football clubs. At the top of the English football league system, it is the countr

In [75]:
def search_documents(query, documents, model, device="cuda"):
    # 질문 임베딩 생성 (GPU로 이동)
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)
    # 문서 임베딩 생성 (GPU로 이동)
    document_embeddings = model.encode(documents, convert_to_tensor=True).to(device)
    
    # 질문과 문서 간 코사인 유사도 계산 (PyTorch 기반)
    query_embedding = query_embedding.unsqueeze(0)  # 배치 차원 추가
    similarities = torch.nn.functional.cosine_similarity(query_embedding, document_embeddings)

    # 유사도에 따라 문서 정렬
    top_results = similarities.argsort(descending=True)[:5]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=documents[idx]
        res[tmp]=similarities[idx]
    return list(res.items())

# 예시 사용법
for i in range(20):
    print(f"Query {i+1} : {question[i]}")
    print("-"*100)
    results = search_documents(question[i], context, model)
    print("Retrieved doc :")
    for j in range(len(results)):
        print(f"\tRank {j} : {results[j]}")
    print("Original doc :", context2[i], "\n")


Query 1 : When does the new bunk'd come out?
----------------------------------------------------------------------------------------------------
tensor([248, 158, 188, 263, 285], device='cuda:0')
Retrieved doc :
	Rank 0 : ('"Luck of the Chuck"Wendy FaraoneMike Montesano & Ted ZizikFebruary\xa012,\xa02021\xa0(2021-02-12)5050.33[102]\nLou takes out Woody the burnt woodchuck as her good luck charm for a camp evaluation, but Destiny and Matteo refuse to believe in luck. Woody is grabbed by a bald eagle during Matteo and Finn\'s bird-watching session. With Woody missing, bad things start happening but Destiny offers to help in order to prove to Lou there is no such thing as bad luck. However, when a deer breaks into her cabin, Destiny falls apart. She becomes very furious at Finn and Matteo after realizing that they\'re the ones who lost Woody, leading to all the bad luck. They later find Woody in a store and Lou wins him back by competing over Moose Rump knowledge.  Meanwhile, Ava agrees 

In [82]:
query = "Who won the football game?"
results = search_documents(query, context, model)
for result in results:
    print(result)

tensor([374, 401, 378, 405, 338], device='cuda:0')
(' 15-yard touchdown reception from Deshaun Watson, 2-point run  failed;38;33;4;1:07;8;75;3:33;ALA;Derrick Henry 1-yard touchdown run, Adam Griffith kick good;45;33;4;0:12;6;68;0:55;CLEM;Jordan Leggett 24-yard touchdown reception from Deshaun Watson, Greg Huegel kick good;45;40;"TOP" = time of possession.  For other American football terms, see Glossary of American football.;45;40\n', tensor(0.8611, device='cuda:0'))
('\n## Broadcasting\nThe game was broadcast in the United States by ESPN, ESPN Deportes, and ESPN Radio, with Chris Fowler and Kirk Herbstreit as English commentators on TV, and Eduardo Varela and Pablo Viruega as Spanish commentators. In Brazil, the game was broadcast on ESPN Brazil by Everaldo Marques (play by play) and Antony Curti (color commentator). As in 2015, ESPN provided Megacast coverage of the game, which supplemented coverage with analysis and additional perspectives of the game on different ESPN channels and 